# 01 · Data exploration

What is actually in the SkillCorner Open Data drop, established by reading the
files rather than the documentation. Everything found here is written up in
`docs/DATA_DICTIONARY.md`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DERIVED_DIR, PROCESSED_DIR, PALETTE
plt.style.use("dark_background")
plt.rcParams.update({
    "figure.facecolor": PALETTE["surface"], "axes.facecolor": PALETTE["surface"],
    "savefig.facecolor": PALETTE["surface"], "grid.color": PALETTE["grid"],
    "axes.edgecolor": PALETTE["line"], "text.color": PALETTE["text"],
    "axes.labelcolor": PALETTE["text_secondary"], "xtick.color": PALETTE["muted"],
    "ytick.color": PALETTE["muted"], "font.size": 9, "figure.dpi": 110,
})
pd.set_option("display.width", 140, "display.max_columns", 40)


## The 20 matches

The repository README says 10; it ships 20.

In [2]:
matches = pd.read_parquet(PROCESSED_DIR / "matches.parquet")
print(f"{len(matches)} matches, {pd.concat([matches.home_team, matches.away_team]).nunique()} teams")
matches[["date", "round", "home_team", "home_score", "away_score", "away_team",
         "pitch_length", "pitch_width", "has_bodypose"]]

20 matches, 13 teams


,date,round,home_team,home_score,away_score,away_team,pitch_length,pitch_width,has_bodypose
0,2024-11-23,Round 5,Brisbane FC,2,3,Adelaide United,105.0,68.0,False
1,2024-11-30,Round 6,Auckland FC,2,0,Newcastle,104.0,68.0,False
2,2024-12-07,Round 7,Auckland FC,2,1,Wellington P FC,104.0,68.0,False
3,2024-12-21,Round 9,Brisbane FC,0,1,Perth Glory,105.0,68.0,True
4,2024-12-22,Round 9,Western Sydney,4,1,Wellington P FC,105.0,68.0,False
5,2024-12-31,Round 11,CC Mariners,1,1,Melbourne City,105.0,68.0,False
6,2025-01-10,Round 13,Melbourne V FC,3,4,Western United,106.0,68.0,False
7,2025-01-26,Round 16,Western Sydney,0,1,Auckland FC,105.0,68.0,False
8,2025-02-01,Round 17,Sydney FC,4,1,Adelaide United,105.0,68.0,True
9,2025-02-01,Round 17,Auckland FC,2,1,Macarthur FC,104.0,68.0,False


Pitch size is not constant, which is why every table also carries coordinates rescaled to a standard 105 x 68 pitch.

In [3]:
matches.groupby(["pitch_length", "pitch_width"]).size().rename("matches").reset_index()

,pitch_length,pitch_width,matches
0,104.0,68.0,4
1,105.0,68.0,10
2,106.0,68.0,6


## The tracking file

One JSON object per video frame at 10 fps.

In [4]:
from src.data.loaders import iter_tracking
import itertools, json
rec = next(itertools.islice(iter_tracking(1874553), 300, 301))
rec["player_data"] = rec["player_data"][:2]
print(json.dumps(rec, indent=1)[:1200])

{
 "frame": 300,
 "timestamp": "00:00:29.00",
 "period": 1,
 "ball_data": {
  "x": -23.81,
  "y": -24.52,
  "z": 0.08,
  "is_detected": true
 },
 "possession": {
  "player_id": null,
  "group": "home team"
 },
 "image_corners_projection": {
  "x_top_left": -87.42,
  "y_top_left": 39.0,
  "x_bottom_left": -27.29,
  "y_bottom_left": -38.81,
  "x_bottom_right": -7.1,
  "y_bottom_right": -29.09,
  "x_top_right": -21.27,
  "y_top_right": 39.0
 },
 "player_data": [
  {
   "x": -41.97,
   "y": -5.15,
   "player_id": 50999,
   "is_detected": true
  },
  {
   "x": -31.24,
   "y": -31.08,
   "player_id": 69111,
   "is_detected": true
  }
 ]
}


In [5]:
from src.preprocessing.build_tracking import load_ball, load_tracking
ball = load_ball(1874553)
trk = load_tracking(1874553)
print(f"frames            {len(ball):,}  ({ball.frame.min()}-{ball.frame.max()}, consecutive: "
      f"{bool((np.diff(np.sort(ball.frame)) == 1).all())})")
print(f"no tracking       {ball.ball_x.isna().sum():,}  ({ball.ball_x.isna().mean():.0%})")
print(f"player-frames     {len(trk):,}")
print(f"detected          {trk.is_detected.mean():.0%}  (the rest are extrapolations)")
print(f"players per frame {sorted(trk.groupby('frame').size().unique())}")

frames            60,301  (0-60300, consecutive: True)
no tracking       16,057  (27%)
player-frames     973,368
detected          58%  (the rest are extrapolations)
players per frame [np.int64(22)]


## The event taxonomy

Four event types share one 322-column table. There is **no** shot, tackle, foul or set-piece event type.

In [6]:
ev = pd.read_parquet(PROCESSED_DIR / "events.parquet",
                     columns=["event_type", "event_subtype"])
ev.value_counts(dropna=False).rename("rows").reset_index()

,event_type,event_subtype,rows
0,passing_option,NaN,48431
1,player_possession,NaN,18792
2,on_ball_engagement,pressure,4622
3,on_ball_engagement,recovery_press,4019
4,on_ball_engagement,pressing,3813
5,on_ball_engagement,other,3324
6,off_ball_run,run_ahead_of_the_ball,2721
7,off_ball_run,support,1623
8,on_ball_engagement,counter_press,1622
9,off_ball_run,coming_short,1312


## Which columns belong to which event type

Most of the 322 columns are empty for most rows, and which ones depends entirely on the event type.

In [7]:
full = pd.read_parquet(PROCESSED_DIR / "events.parquet")
probe = ["x_start", "possession_epv_total", "overall_pressure_start", "pass_outcome",
         "first_line_break", "event_subtype", "pressing_chain", "player_targeted_id",
         "xthreat", "xpass_completion", "player_targeted_xpass_completion"]
(full.groupby("event_type")[probe].apply(lambda g: g.notna().mean()).round(2))

,x_start,possession_epv_total,overall_pressure_start,pass_outcome,first_line_break,event_subtype,pressing_chain,player_targeted_id,xthreat,xpass_completion,player_targeted_xpass_completion
event_type,,,,,,,,,,,
off_ball_run,1.0,0.00,0.00,0.00,0.00,1.0,0.0,0.00,1.0,1.0,0.00
on_ball_engagement,1.0,0.00,0.00,0.00,0.00,1.0,1.0,1.00,0.0,0.0,0.00
passing_option,1.0,0.00,0.00,0.00,0.44,0.0,0.0,0.00,1.0,1.0,0.00
player_possession,1.0,0.76,0.71,0.87,0.39,0.0,0.0,0.87,0.0,0.0,0.81


`xthreat` and `xpass_completion` sit on `passing_option` and `off_ball_run`
rows and are **never** populated on `player_possession`. The possession
equivalents are the `player_targeted_*` columns. Getting this wrong silently
produces an all-NaN column.

## Coverage gaps in the source

Four matches carry no EPV, pressure or reception difficulty at all.

In [8]:
poss = pd.read_parquet(DERIVED_DIR / "possessions.parquet",
                       columns=["match_id", "possession_epv_total", "overall_pressure_start",
                                "reception_difficulty_start"])
cov = poss.groupby("match_id").apply(
    lambda g: pd.Series({"possessions": len(g),
                         "epv": g.possession_epv_total.notna().mean(),
                         "pressure": g.overall_pressure_start.notna().mean(),
                         "reception_difficulty": g.reception_difficulty_start.notna().mean()}),
    include_groups=False).round(3)
cov

,possessions,epv,pressure,reception_difficulty
match_id,,,,
1874553,959.0,0.967,0.902,0.885
1886347,999.0,0.000,0.000,0.000
1899585,929.0,0.000,0.000,0.000
1925299,1062.0,0.000,0.000,0.000
1927964,1140.0,0.968,0.926,0.916
1953632,979.0,0.968,0.910,0.895
1959846,946.0,0.920,0.887,0.870
1986691,915.0,0.966,0.885,0.867
1996435,1005.0,0.965,0.894,0.881


In [9]:
missing = cov[cov.epv == 0].index.tolist()
print("matches with no EPV at all:", missing)
print(f"{poss.match_id.isin(missing).sum():,} of {len(poss):,} possessions "
      f"({poss.match_id.isin(missing).mean():.0%})")

matches with no EPV at all: [1886347, 1899585, 1925299, 2017461]
3,829 of 18,792 possessions (20%)


## Phases of play

In [10]:
ph = pd.read_parquet(PROCESSED_DIR / "phases.parquet")
print("in possession :", sorted(ph.team_in_possession_phase_type.dropna().unique()))
print("out of poss.  :", sorted(ph.team_out_of_possession_phase_type.dropna().unique()))
ph.groupby("team_in_possession_phase_type").agg(
    n=("duration", "size"), mean_duration_s=("duration", "mean")).round(1)

in possession : ['build_up', 'chaotic', 'create', 'direct', 'finish', 'quick_break', 'set_play', 'transition']
out of poss.  : ['chaotic', 'defending_direct', 'defending_quick_break', 'defending_set_play', 'defending_transition', 'high_block', 'low_block', 'medium_block']


,n,mean_duration_s
team_in_possession_phase_type,,
build_up,1190,8.6
chaotic,1877,3.3
create,2869,8.6
direct,683,3.3
finish,1606,8.6
quick_break,106,8.8
set_play,354,7.0
transition,189,12.1
